# Construction-PPE Training (Kaggle GPU)

Train the 11-class PPE model on a free Kaggle GPU, then download `ppe_model.pt` and use it locally.

**Before running:** Settings (right sidebar) -> Accelerator = **GPU**, Internet = **ON**.

Classes: `helmet, gloves, vest, boots, goggles, none, Person, no_helmet, no_goggle, no_gloves, no_boots`

In [ ]:
# Install ultralytics pinned to your LOCAL version so weights load cleanly.
# Local version = 8.4.170 (change if yours differs).
!pip install -q ultralytics==8.4.170
import ultralytics, torch
print('ultralytics:', ultralytics.__version__)
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from ultralytics import YOLO

BASE_MODEL = 'yolo26m.pt'   # strong at people + PPE; use 'yolo11m.pt' if unavailable
EPOCHS = 25
IMGSZ  = 640
BATCH  = 16

model = YOLO(BASE_MODEL)
results = model.train(
    data='construction-ppe.yaml',   # auto-downloads (~178 MB)
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0, name='ppe_construction_gpu',
    patience=15, plots=True, cache=True,
)

In [ ]:
# Validate and show accuracy
metrics = model.val()
print('mAP50:   ', round(float(metrics.box.map50), 4))
print('mAP50-95:', round(float(metrics.box.map), 4))
print('Classes: ', model.names)

In [ ]:
# Export weights for download (appears in the Output tab, right panel)
import shutil, os
best = model.trainer.best
out = '/kaggle/working/ppe_model.pt'
shutil.copyfile(best, out)
print('Saved:', out, f'({os.path.getsize(out)/1e6:.1f} MB)')
print('Download from Output tab -> place in project as ppe_model.pt -> python main.py --source 0')